In [4]:
"""
DataQualityCode.py — Reproducible Front-to-Back Data Quality Pipeline
Fall into BC 2026 Case Competition (Snapshot: 10 July 2026)

Regenerates all defect counts, look-alike clearances, position reconstructions,
indicative exposures, domain/platform scores, Dirichlet sensitivity bounds,
stress-test metrics, and scorecard JSON artifacts from the 10 raw tables.
"""

import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

# =============================================================================
# 1. CENTRALIZED CONFIGURATION (Editable for Live Q&A)
# =============================================================================
CONFIG = {
    "data_dir": Path("."),
    "output_dir": Path("outputs"),
    "snapshot_date": "2026-07-10",
    "seed_weights": 42,
    "seed_stress": 7,
    "dirichlet_draws": 500,
    "stress_sample_accounts": 5000,
    # Tolerances & Rules
    "recon_qty_tol": 0.01,          # Units tolerance for position reconciliation
    "gross_amt_tol": 0.01,          # 1% relative tolerance for qty * price vs gross_amount
    "price_disp_tol": 0.005,        # 0.5% cross-vendor dispersion tolerance
    "price_spike_ratio": 50.0,      # Threshold for 100x single-vendor spike detection
    "nav_bp_tol": 0.0001,           # 1 bp (0.01%) NAV arithmetic tolerance
    "settle_min_days": 1,           # T+1 minimum standard settlement
    "settle_max_days": 4,           # T+4 maximum standard settlement
    "vendor_priority": None,        # Auto-detected sorted vendors unless overridden
    "valid_iso_domiciles": {"US", "GB", "CA", "DE", "FR", "JP", "SG", "CH", "LU", "IE", "AU", "NL"},
    # Domain Weights (Sum = 1.00)
    "domain_weights": {
        "accounts": 0.10,
        "securities": 0.10,
        "transactions": 0.20,
        "pipeline": 0.10,
        "positions": 0.20,
        "pricing": 0.10,
        "corporate_actions": 0.10,
        "ledger": 0.10,
    },
}


# =============================================================================
# 2. DATA LOADING (Raw Text Preserved; Typed Copies Created Downstream)
# =============================================================================
def load_tables(data_dir: Path) -> dict[str, pd.DataFrame]:
    files = {
        "accounts": "accounts.csv.gz",
        "securities": "securities.csv.gz",
        "transactions": "transactions.csv.gz",
        "corporate_actions": "corporate_actions.csv.gz",
        "prices": "prices.csv.gz",
        "positions": "positions.csv.gz",
        "comparison_positions": "comparison_positions.csv.gz",
        "pipeline_events": "pipeline_events.csv.gz",
        "ledger_nav": "ledger_nav.csv.gz",
        "fx_rates": "fx_rates.csv.gz",
    }
    tables = {}
    for name, fname in files.items():
        path = data_dir / fname
        if not path.exists():
            path = data_dir / fname.replace(".gz", "")
        tables[name] = pd.read_csv(path, dtype=str, keep_default_na=True)
    return tables


def parse_mixed_date(series: pd.Series) -> pd.Series:
    """Parses ISO YYYY-MM-DD and DD/MM/YYYY dates safely without silent coercion."""
    s = series.astype("string").str.strip()
    iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    dmy_mask = s.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    out.loc[iso_mask] = pd.to_datetime(s.loc[iso_mask], format="%Y-%m-%d", errors="coerce")
    out.loc[dmy_mask] = pd.to_datetime(s.loc[dmy_mask], format="%d/%m/%Y", errors="coerce")
    return out


# =============================================================================
# 3. MASTER DATA CHECKS (M01 - M06)
# =============================================================================
def check_masters(acc: pd.DataFrame, sec: pd.DataFrame, fx: pd.DataFrame) -> dict:
    m01 = acc["base_currency"].isna() | (acc["base_currency"].str.strip() == "")
    m02 = ~acc["domicile"].isin(CONFIG["valid_iso_domiciles"])

    m03 = sec["isin"].notna() & sec.duplicated("isin", keep=False)
    m04 = sec["cusip"].notna() & sec.duplicated("cusip", keep=False)
    m05 = sec["cusip"].isna() | (sec["cusip"].str.strip() == "")
    valid_fx_ccy = set(fx["currency"].dropna().unique())
    m06 = ~sec["currency"].isin(valid_fx_ccy)

    return {
        "M01": {"affected": int(m01.sum()), "eligible": len(acc)},
        "M02": {"affected": int(m02.sum()), "eligible": len(acc)},
        "M03": {"affected": int(m03.sum()), "eligible": len(sec)},
        "M04": {"affected": int(m04.sum()), "eligible": len(sec)},
        "M05": {"affected": int(m05.sum()), "eligible": len(sec)},
        "M06": {"affected": int(m06.sum()), "eligible": len(sec)},
        "acc_clean_rate": 1.0 - float((m01 | m02).mean()),
        "sec_clean_rate": 1.0 - float((m03 | m04 | m05 | m06).mean()),
    }


# =============================================================================
# 4. TRANSACTION & PIPELINE CHECKS (T01 - T14, D01 - D04) & LOOK-ALIKES
# =============================================================================
def check_transactions_and_pipeline(
    txn: pd.DataFrame, pip: pd.DataFrame, acc: pd.DataFrame, sec: pd.DataFrame
) -> tuple[dict, pd.DataFrame]:
    df = txn.copy()
    df["qty_num"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["prc_num"] = pd.to_numeric(df["price"], errors="coerce")
    df["grs_num"] = pd.to_numeric(df["gross_amount"], errors="coerce")
    df["td_dt"] = parse_mixed_date(df["trade_date"])
    df["sd_dt"] = parse_mixed_date(df["settlement_date"])
    df["evt_dt"] = pd.to_datetime(df["event_timestamp"], errors="coerce")

    # Look-alike 1: DD/MM/YYYY trade dates verified against event_timestamp
    dmy_mask = df["trade_date"].astype("string").str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
    dmy_cleared = dmy_mask & (df["td_dt"].dt.date == df["evt_dt"].dt.date)

    # T01 & T02: Duplicate transaction_ids
    dup_id_mask = df.duplicated("transaction_id", keep="first")
    exact_dup_mask = df.duplicated(subset=txn.columns, keep="first")
    conflict_dup_mask = dup_id_mask & (~exact_dup_mask)

    # T03, T04, T05, T06: Completeness, Quantity, Settlement, 100x Price
    t03 = df["qty_num"].isna() | df["prc_num"].isna() | df["settlement_date"].isna()
    t04 = df["qty_num"] <= 0

    bday_lag = np.busday_count(
        df["td_dt"].fillna(pd.Timestamp("1970-01-01")).values.astype("datetime64[D]"),
        df["sd_dt"].fillna(pd.Timestamp("1970-01-01")).values.astype("datetime64[D]"),
    )
    valid_dates = df["td_dt"].notna() & df["sd_dt"].notna()
    note_cols = [c for c in df.columns if "note" in c.lower() or "comment" in c.lower() or "ext" in c.lower()]
    if note_cols:
        has_note = df[note_cols[0]].notna() & (df[note_cols[0]].str.strip() != "")
    else:
        has_note = pd.Series(False, index=df.index)

    ext_settle_cleared = valid_dates & (bday_lag == 7) & has_note
    t05 = valid_dates & ((bday_lag < CONFIG["settle_min_days"]) | (bday_lag > CONFIG["settle_max_days"])) & (~ext_settle_cleared)

    expected_gross = df["qty_num"].abs() * df["prc_num"].abs()
    t06 = (
        (df["qty_num"].abs() > 0)
        & (df["prc_num"] > 0)
        & (df["grs_num"].abs() > 0)
        & (((df["grs_num"].abs() - expected_gross * 0.01).abs() / (expected_gross * 0.01)) < CONFIG["gross_amt_tol"])
    )

    # T07, T08, T14: Lifecycle, Orphan Cancels, Double Reversals, Rebooks
    all_ids = set(df["transaction_id"].dropna())
    orig_lookup = df.drop_duplicates("transaction_id").set_index("transaction_id")

    is_cancel = df["status"] == "CANCELLED"
    has_rel = df["related_transaction_id"].notna() & (df["related_transaction_id"].str.strip() != "")
    t07 = is_cancel & has_rel & (~df["related_transaction_id"].isin(all_ids))

    reversals = df[df["status"] == "REVERSED"]
    rev_counts = reversals.groupby("related_transaction_id").size()
    double_rev_orig_ids = set(rev_counts[rev_counts == 2].index)

    # Rebooks: BOOKED row linked to a CANCELLED original
    rebook_mask = (df["status"] == "BOOKED") & df["related_transaction_id"].isin(all_ids)
    rebook_orig_qty = df.loc[rebook_mask, "related_transaction_id"].map(orig_lookup["qty_num"])
    rebook_orig_grs = df.loc[rebook_mask, "related_transaction_id"].map(orig_lookup["grs_num"])
    t14_sub = (
        (df.loc[rebook_mask, "qty_num"] != rebook_orig_qty)
        & ((df.loc[rebook_mask, "grs_num"] - rebook_orig_grs).abs() < 0.01)
    )
    t14 = pd.Series(False, index=df.index)
    t14.loc[rebook_mask] = t14_sub

    # T09 - T13: Master Referential Integrity & Closed Accounts
    sec_map = sec.drop_duplicates("security_id").set_index("security_id")
    acc_map = acc.drop_duplicates("account_id").set_index("account_id")

    t11 = ~df["security_id"].isin(sec_map.index)
    t12 = ~df["account_id"].isin(acc_map.index)

    m_cusip = df["security_id"].map(sec_map["cusip"])
    m_ccy = df["security_id"].map(sec_map["currency"])
    t09 = (~t11) & (df["cusip"].fillna("") != m_cusip.fillna(""))
    t10 = (~t11) & (df["currency"] != m_ccy)

    acc_close_dt = parse_mixed_date(df["account_id"].map(acc_map["close_date"]))
    t13 = (~t12) & acc_close_dt.notna() & df["td_dt"].notna() & (df["td_dt"] > acc_close_dt)

    # Pipeline Timeliness (D01 - D04)
    p_rec = pd.to_datetime(pip["received_timestamp"], errors="coerce")
    p_bok = pd.to_datetime(pip["booked_timestamp"], errors="coerce")
    p_pub = pd.to_datetime(pip["published_timestamp"], errors="coerce")
    p_cut = pd.to_datetime(pip["cutoff_timestamp"], errors="coerce")

    d01 = p_rec.isna()
    d02 = p_rec.notna() & p_bok.notna() & (p_bok < p_rec)
    d03 = p_pub.notna() & p_cut.notna() & (p_pub > p_cut)
    d04 = pip.duplicated("transaction_id", keep="first")

    any_txn_defect = (
        dup_id_mask | t03 | t04 | t05 | t06 | t07 | t09 | t10 | t11 | t12 | t13
    )
    any_pip_defect = d01 | d02 | d03 | d04

    results = {
        "T01": {"affected": int(exact_dup_mask.sum()), "eligible": len(df)},
        "T02": {"affected": int(conflict_dup_mask.sum()), "eligible": len(df)},
        "T03": {"affected": int(t03.sum()), "eligible": len(df)},
        "T04": {"affected": int(t04.sum()), "eligible": len(df)},
        "T05": {"affected": int(t05.sum()), "eligible": len(df)},
        "T06": {"affected": int(t06.sum()), "eligible": len(df)},
        "T07": {"affected": int(t07.sum()), "eligible": len(df)},
        "T08": {"affected": int(len(double_rev_orig_ids)), "eligible": int(len(rev_counts))},
        "T09": {"affected": int(t09.sum()), "eligible": len(df)},
        "T10": {"affected": int(t10.sum()), "eligible": len(df)},
        "T11": {"affected": int(t11.sum()), "eligible": len(df)},
        "T12": {"affected": int(t12.sum()), "eligible": len(df)},
        "T13": {"affected": int(t13.sum()), "eligible": len(df)},
        "T14": {"affected": int(t14.sum()), "eligible": int(rebook_mask.sum())},
        "D01": {"affected": int(d01.sum()), "eligible": len(pip)},
        "D02": {"affected": int(d02.sum()), "eligible": int((~d01).sum())},
        "D03": {"affected": int(d03.sum()), "eligible": len(pip)},
        "D04": {"affected": int(d04.sum()), "eligible": len(pip)},
        "lookalike_dmy_cleared": int(dmy_cleared.sum()),
        "lookalike_ext_settle_cleared": int(ext_settle_cleared.sum()),
        "txn_clean_rate": 1.0 - float(any_txn_defect.mean()),
        "pip_clean_rate": 1.0 - float(any_pip_defect.mean()),
    }
    return results, df


# =============================================================================
# 5. PRICING, FX & CORPORATE ACTIONS (P01 - P06, X01, C01 - C03)
# =============================================================================
def check_pricing_fx_ca(
    prc: pd.DataFrame, sec: pd.DataFrame, fx: pd.DataFrame, ca: pd.DataFrame
) -> tuple[dict, pd.Series]:
    p = prc.copy()
    p["px"] = pd.to_numeric(p["price"], errors="coerce")
    sec_ccy = sec.drop_duplicates("security_id").set_index("security_id")["currency"]

    p01 = p["px"].isna()
    p02 = p["px"].notna() & (p["px"] <= 0)
    p03 = (p["currency"] != p["security_id"].map(sec_ccy)) if "currency" in p.columns else pd.Series(False, index=p.index)

    # Identify date and vendor columns dynamically
    date_col = "price_date" if "price_date" in p.columns else [c for c in p.columns if "date" in c][0]
    vendor_col = "vendor" if "vendor" in p.columns else [c for c in p.columns if "vendor" in c or "source" in c][0]

    # Security-day pivot across all vendors
    valid_p = p[p["px"].notna() & (p["px"] > 0)].copy()
    piv = valid_p.pivot_table(
        index=["security_id", date_col], columns=vendor_col, values="px", aggfunc="first"
    )
    v_min = piv.min(axis=1)
    v_max = piv.max(axis=1)
    v_mean = piv.mean(axis=1)
    v_med = piv.median(axis=1)

    p04 = ((v_max - v_min) / v_mean) > CONFIG["price_disp_tol"]
    p05 = (v_max / v_med) > CONFIG["price_spike_ratio"]

    # Stale primary vendor (first priority vendor unchanged while at least one peer moved)
    piv_sorted = piv.sort_index()
    vendors = CONFIG["vendor_priority"] or sorted(piv_sorted.columns.tolist())
    primary_v = vendors[0]
    peer_vs = [v for v in vendors[1:] if v in piv_sorted.columns]

    prim_diff = piv_sorted.groupby(level=0)[primary_v].diff().abs()
    peer_moved = pd.Series(False, index=piv_sorted.index)
    for pv in peer_vs:
        peer_moved = peer_moved | (piv_sorted.groupby(level=0)[pv].diff().abs() > 1e-6)
    p06 = (prim_diff == 0) & peer_moved

    # Snapshot median vendor price per security
    snap_px = (
        valid_p[valid_p[date_col] == CONFIG["snapshot_date"]]
        .groupby("security_id")["px"]
        .median()
    )

    # FX USD=1.0 check (X01)
    rate_col = "rate_to_usd" if "rate_to_usd" in fx.columns else [c for c in fx.columns if "rate" in c][0]
    usd_rows = fx[fx["currency"] == "USD"].copy()
    usd_rows["rate"] = pd.to_numeric(usd_rows[rate_col], errors="coerce")
    x01 = (usd_rows["rate"] - 1.0).abs() > 1e-6

    # Corporate actions (C01 - C03)
    ex_dt = parse_mixed_date(ca["ex_date"])
    eff_dt = parse_mixed_date(ca["effective_date"])
    c01 = eff_dt < ex_dt

    sec_ids = set(sec["security_id"].dropna())
    type_col = "action_type" if "action_type" in ca.columns else [c for c in ca.columns if "type" in c][0]
    target_col = "target_security_id" if "target_security_id" in ca.columns else [c for c in ca.columns if "target" in c][0]
    needs_target = ca[type_col].isin(["SPINOFF", "MERGER", "STOCK_SPLIT", "NAME_CHANGE"])
    c02 = needs_target & (~ca[target_col].isin(sec_ids))

    ratio_col = "ratio" if "ratio" in ca.columns else [c for c in ca.columns if "ratio" in c][0]
    cash_col = "cash_amount" if "cash_amount" in ca.columns else [c for c in ca.columns if "cash" in c][0]
    terms_missing = ca[ratio_col].isna() & ca[cash_col].isna()
    c03 = c01 | c02 | terms_missing

    results = {
        "P01": {"affected": int(p01.sum()), "eligible": len(p)},
        "P02": {"affected": int(p02.sum()), "eligible": len(p)},
        "P03": {"affected": int(p03.sum()), "eligible": len(p)},
        "P04": {"affected": int(p04.sum()), "eligible": len(piv)},
        "P05": {"affected": int(p05.sum()), "eligible": len(piv)},
        "P06": {"affected": int(p06.sum()), "eligible": len(piv)},
        "X01": {"affected": int(x01.sum()), "eligible": len(usd_rows)},
        "C01": {"affected": int(c01.sum()), "eligible": len(ca)},
        "C02": {"affected": int(c02.sum()), "eligible": len(ca)},
        "C03": {"affected": int(c03.sum()), "eligible": len(ca)},
        "prc_clean_rate": 1.0 - float(p04.mean()),
        "ca_clean_rate": 1.0 - float(c03.mean()),
    }
    return results, snap_px


# =============================================================================
# 6. POSITION RECONCILIATION (A-E), LEDGER CHECKS & INDICATIVE EXPOSURE
# =============================================================================
def check_positions_ledger_exposure(
    txn_df: pd.DataFrame,
    pos: pd.DataFrame,
    cmp: pd.DataFrame,
    ldg: pd.DataFrame,
    sec: pd.DataFrame,
    fx: pd.DataFrame,
    snap_px: pd.Series,
) -> dict:
    p_df = pos.copy()
    c_df = cmp.copy()
    p_df["qty_book"] = pd.to_numeric(p_df["quantity"], errors="coerce")
    c_df["qty_cust"] = pd.to_numeric(c_df["quantity"], errors="coerce")

    # Full-union book vs custodian reconciliation (R01, R02)
    recon = p_df.merge(c_df, on=["account_id", "security_id"], how="outer")
    in_both = recon[recon["qty_book"].notna() & recon["qty_cust"].notna()]
    r01 = in_both[(in_both["qty_book"] - in_both["qty_cust"]).abs() >= CONFIG["recon_qty_tol"]]
    r02_book_only = recon["qty_cust"].isna().sum()
    r02_cust_only = recon["qty_book"].isna().sum()

    # Signed transaction rebuild helper
    def build_positions(df_in: pd.DataFrame, dedupe: bool, rev_mode: str, date_col: str) -> pd.DataFrame:
        sub = df_in[df_in["status"] != "CANCELLED"].copy()
        sub = sub[sub[date_col].notna() & (sub[date_col] <= pd.Timestamp(CONFIG["snapshot_date"]))]
        if dedupe:
            sub = sub.drop_duplicates("transaction_id", keep="first")
        qty = sub["qty_num"].fillna(0.0)
        sign = np.where(sub["transaction_type"] == "BUY", 1.0, -1.0)

        if rev_mode == "booked_only":
            sub = sub[sub["status"] == "BOOKED"]
            qty = sub["qty_num"].fillna(0.0)
            sign = np.where(sub["transaction_type"] == "BUY", 1.0, -1.0)
            signed = qty * sign
        elif rev_mode == "offset_all":
            rev_flip = np.where(sub["status"] == "REVERSED", -1.0, 1.0)
            signed = qty * sign * rev_flip
        elif rev_mode == "offset_once":
            rev_rows = sub[sub["status"] == "REVERSED"].drop_duplicates("related_transaction_id", keep="first")
            non_rev = sub[sub["status"] != "REVERSED"]
            sub = pd.concat([non_rev, rev_rows], ignore_index=True)
            qty = sub["qty_num"].fillna(0.0)
            sign = np.where(sub["transaction_type"] == "BUY", 1.0, -1.0)
            rev_flip = np.where(sub["status"] == "REVERSED", -1.0, 1.0)
            signed = qty * sign * rev_flip
        else:
            raise ValueError(f"Unknown rev_mode: {rev_mode}")

        sub["signed_qty"] = signed
        agg = sub.groupby(["account_id", "security_id"], as_index=False)["signed_qty"].sum()
        return agg[agg["signed_qty"].abs() >= 1e-6]

    # Five Interpretations (A - E)
    interp_specs = {
        "A_booked_dedupe": build_positions(txn_df, dedupe=True, rev_mode="booked_only", date_col="td_dt"),
        "B_book_replica": build_positions(txn_df, dedupe=False, rev_mode="offset_all", date_col="td_dt"),
        "C_dedupe_rev_all": build_positions(txn_df, dedupe=True, rev_mode="offset_all", date_col="td_dt"),
        "D_dedupe_rev_once": build_positions(txn_df, dedupe=True, rev_mode="offset_once", date_col="td_dt"),
        "E_replica_settle": build_positions(txn_df, dedupe=False, rev_mode="offset_all", date_col="sd_dt"),
    }

    interp_summary = {}
    for name, hist in interp_specs.items():
        m_rep = p_df.merge(hist, on=["account_id", "security_id"], how="inner")
        b_rep = ((m_rep["qty_book"] - m_rep["signed_qty"]).abs() >= CONFIG["recon_qty_tol"]).sum()
        m_cst = c_df.merge(hist, on=["account_id", "security_id"], how="inner")
        b_cst = ((m_cst["qty_cust"] - m_cst["signed_qty"]).abs() >= CONFIG["recon_qty_tol"]).sum()
        interp_summary[name] = {
            "vs_reported_match_pct": round(100.0 * (1.0 - b_rep / len(m_rep)), 3),
            "vs_reported_breaks": int(b_rep),
            "vs_custodian_match_pct": round(100.0 * (1.0 - b_cst / len(m_cst)), 3),
            "vs_custodian_breaks": int(b_cst),
        }

    # R03: Book vs Interpretation B breaks
    hist_b = interp_specs["B_book_replica"]
    b_vs_h = p_df.merge(hist_b, on=["account_id", "security_id"], how="inner")
    r03 = b_vs_h[(b_vs_h["qty_book"] - b_vs_h["signed_qty"]).abs() >= CONFIG["recon_qty_tol"]]
    sign_inv = r03[(r03["qty_book"] + r03["signed_qty"]).abs() < CONFIG["recon_qty_tol"]]

    # Indicative Gross Exposure (USD)
    fx_date_col = "rate_date" if "rate_date" in fx.columns else [c for c in fx.columns if "date" in c][0]
    fx_rate_col = "rate_to_usd" if "rate_to_usd" in fx.columns else [c for c in fx.columns if "rate" in c][0]
    snap_fx = (
        fx[fx[fx_date_col] == CONFIG["snapshot_date"]]
        .assign(rate=lambda d: np.where(d["currency"] == "USD", 1.0, pd.to_numeric(d[fx_rate_col], errors="coerce")))
        .set_index("currency")["rate"]
    )
    sec_ccy = sec.drop_duplicates("security_id").set_index("security_id")["currency"]

    def usd_unit_value(sec_series: pd.Series) -> pd.Series:
        px = sec_series.map(snap_px).fillna(0.0)
        ccy = sec_series.map(sec_ccy)
        rate = ccy.map(snap_fx).fillna(1.0)
        return px * rate

    # 1. Duplicate transaction position impact (B vs C)
    hist_c = interp_specs["C_dedupe_rev_all"]
    dup_pos = p_df.merge(hist_c, on=["account_id", "security_id"], how="inner")
    dup_diff = dup_pos[(dup_pos["qty_book"] - dup_pos["signed_qty"]).abs() >= CONFIG["recon_qty_tol"]].copy()
    dup_diff["unit_diff"] = (dup_diff["qty_book"] - dup_diff["signed_qty"]).abs()
    dup_diff["usd_exp"] = dup_diff["unit_diff"] * usd_unit_value(dup_diff["security_id"])

    # 2. History break exposure (R03)
    r03_exp = r03.copy()
    r03_exp["unit_diff"] = (r03_exp["qty_book"] - r03_exp["signed_qty"]).abs()
    r03_exp["usd_exp"] = r03_exp["unit_diff"] * usd_unit_value(r03_exp["security_id"])

    # 3. Double reversal exposure (C vs D)
    hist_d = interp_specs["D_dedupe_rev_once"]
    rev_cmp = hist_c.merge(hist_d, on=["account_id", "security_id"], suffixes=("_c", "_d"))
    rev_diff = rev_cmp[(rev_cmp["signed_qty_c"] - rev_cmp["signed_qty_d"]).abs() >= CONFIG["recon_qty_tol"]].copy()
    rev_diff["unit_diff"] = (rev_diff["signed_qty_c"] - rev_diff["signed_qty_d"]).abs()
    rev_diff["usd_exp"] = rev_diff["unit_diff"] * usd_unit_value(rev_diff["security_id"])

    # Ledger Arithmetic (L01, L02)
    l_df = ldg.copy()
    l_df["na"] = pd.to_numeric(l_df["net_assets"], errors="coerce")
    l_df["sh"] = pd.to_numeric(l_df["shares_outstanding"], errors="coerce")
    l_df["nav"] = pd.to_numeric(l_df["nav_per_share"], errors="coerce")
    l01 = (((l_df["na"] / l_df["sh"]) - l_df["nav"]).abs() / l_df["nav"].abs()) > CONFIG["nav_bp_tol"]
    l02 = l_df["na"] < 0

    return {
        "R01": {"affected": int(len(r01)), "eligible": int(len(in_both))},
        "R02_book_only": int(r02_book_only),
        "R02_cust_only": int(r02_cust_only),
        "R03": {"affected": int(len(r03)), "eligible": int(len(b_vs_h)), "sign_inversions": int(len(sign_inv))},
        "L01": {"affected": int(l01.sum()), "eligible": len(l_df)},
        "L02": {"affected": int(l02.sum()), "eligible": len(l_df)},
        "interpretations": interp_summary,
        "exposures_usd": {
            "duplicates_positions": int(len(dup_diff)),
            "duplicates_accounts": int(dup_diff["account_id"].nunique()),
            "duplicates_units": round(float(dup_diff["unit_diff"].sum()), 0),
            "duplicates_usd": round(float(dup_diff["usd_exp"].sum()), 0),
            "history_breaks_positions": int(len(r03_exp)),
            "history_breaks_accounts": int(r03_exp["account_id"].nunique()),
            "history_breaks_units": round(float(r03_exp["unit_diff"].sum()), 0),
            "history_breaks_usd": round(float(r03_exp["usd_exp"].sum()), 0),
            "double_rev_positions": int(len(rev_diff)),
            "double_rev_accounts": int(rev_diff["account_id"].nunique()),
            "double_rev_units": round(float(rev_diff["unit_diff"].sum()), 0),
            "double_rev_usd": round(float(rev_diff["usd_exp"].sum()), 0),
        },
        "pos_clean_rate": 1.0 - float((len(r01) + r02_book_only + len(r03)) / len(recon)),
        "ldg_clean_rate": 1.0 - float((l01 | l02).mean()),
    }


# =============================================================================
# 7. SCORING, DIRICHLET SENSITIVITY & STRESS TEST + ROW-ORDER INVARIANCE
# =============================================================================
def compute_scores_and_sensitivity(domain_rates: dict[str, float]) -> dict:
    weights = CONFIG["domain_weights"]
    platform_score = sum(domain_rates[d] * weights[d] for d in weights) * 100.0

    rng = np.random.default_rng(CONFIG["seed_weights"])
    alpha = np.array([weights[d] * 20.0 for d in weights])
    draws = rng.dirichlet(alpha, size=CONFIG["dirichlet_draws"])
    rate_vec = np.array([domain_rates[d] for d in weights])
    sim_scores = (draws @ rate_vec) * 100.0

    return {
        "domain_scores_pct": {k: round(v * 100.0, 1) for k, v in domain_rates.items()},
        "platform_score_pct": round(float(platform_score), 1),
        "dirichlet_p05": round(float(np.percentile(sim_scores, 5)), 1),
        "dirichlet_p95": round(float(np.percentile(sim_scores, 95)), 1),
    }


def run_stress_test(txn: pd.DataFrame, acc: pd.DataFrame) -> dict:
    """Injects 210 defects (60 dup IDs, 60 orphan cancels, 60 double revs, 30 account swaps)
    and 120 valid look-alikes (60 valid rebooks, 60 agreed extended settlements),
    evaluating both recall and false positives using the actual detection rules."""
    rng = np.random.default_rng(CONFIG["seed_stress"])
    sample_accs = set(rng.choice(acc["account_id"].unique(), size=CONFIG["stress_sample_accounts"], replace=False))
    base = txn[txn["account_id"].isin(sample_accs)].copy().reset_index(drop=True)

    booked = base[base["status"] == "BOOKED"]
    pick_idx = rng.choice(booked.index, size=330, replace=False)

    # 1. Inject 60 duplicate IDs
    dup_rows = base.loc[pick_idx[:60]].copy()

    # 2. Inject 60 orphan cancels
    orphan_rows = base.loc[pick_idx[60:120]].copy()
    orphan_rows["transaction_id"] = [f"STRESS_ORPH_{i}" for i in range(60)]
    orphan_rows["status"] = "CANCELLED"
    orphan_rows["related_transaction_id"] = [f"MISSING_ORIG_{i}" for i in range(60)]

    # 3. Inject 60 double reversals
    rev_rows_1 = base.loc[pick_idx[120:180]].copy()
    rev_rows_1["transaction_id"] = [f"STRESS_REV1_{i}" for i in range(60)]
    rev_rows_1["status"] = "REVERSED"
    rev_rows_1["related_transaction_id"] = base.loc[pick_idx[120:180], "transaction_id"].values
    rev_rows_2 = rev_rows_1.copy()
    rev_rows_2["transaction_id"] = [f"STRESS_REV2_{i}" for i in range(60)]

    # 4. Inject 30 silent account swaps (row-level rules cannot catch without external order log)
    swapped = base.copy()
    swap_idx = pick_idx[180:210]
    swapped.loc[swap_idx, "account_id"] = rng.permutation(swapped.loc[swap_idx, "account_id"].values)

    # 5a. Valid look-alike 1: 60 valid rebooks (BOOKED pointing to an existing CANCELLED original)
    orig_to_cancel = swapped.loc[pick_idx[210:270], "transaction_id"].values
    swapped.loc[pick_idx[210:270], "status"] = "CANCELLED"
    valid_rebooks = base.loc[pick_idx[210:270]].copy()
    valid_rebooks["transaction_id"] = [f"STRESS_RBK_{i}" for i in range(60)]
    valid_rebooks["status"] = "BOOKED"
    valid_rebooks["related_transaction_id"] = orig_to_cancel

    # 5b. Valid look-alike 2: 60 agreed T+7 extended settlements with agreement note
    valid_ext = base.loc[pick_idx[270:330]].copy()
    valid_ext["transaction_id"] = [f"STRESS_EXT_{i}" for i in range(60)]
    td_ext = parse_mixed_date(valid_ext["trade_date"])
    valid_ext["settlement_date"] = (td_ext + pd.offsets.BDay(7)).dt.strftime("%Y-%m-%d")
    note_col = next(
        (c for c in valid_ext.columns if "note" in c.lower() or "comment" in c.lower() or "ext" in c.lower()),
        "notes",
    )
    valid_ext[note_col] = "AGREED_T+7_EXT"

    stressed = pd.concat(
        [swapped, dup_rows, orphan_rows, rev_rows_1, rev_rows_2, valid_rebooks, valid_ext],
        ignore_index=True,
    )

    # --- Evaluate Defect Recall (on 210 injected defects) ---
    all_ids = set(stressed["transaction_id"].dropna())
    det_dups = stressed[stressed.duplicated("transaction_id", keep="first")]["transaction_id"].isin(
        set(dup_rows["transaction_id"])
    ).sum()
    det_orph = (
        (stressed["status"] == "CANCELLED")
        & (~stressed["related_transaction_id"].isin(all_ids))
        & stressed["transaction_id"].str.startswith("STRESS_ORPH_")
    ).sum()
    rev_grp = stressed[stressed["status"] == "REVERSED"].groupby("related_transaction_id").size()
    det_drev = sum(1 for oid in rev_rows_1["related_transaction_id"] if rev_grp.get(oid, 0) == 2)

    caught = int(det_dups + det_orph + det_drev)
    missed = 210 - caught

    # --- Evaluate False Positives (on 120 valid look-alikes) ---
    # Check 1: Are any valid rebooks flagged as duplicate IDs, orphan links, or double reversals?
    rbk_mask = stressed["transaction_id"].str.startswith("STRESS_RBK_", na=False)
    rbk_dup_fp = stressed.duplicated("transaction_id", keep="first") & rbk_mask
    rbk_link_fp = rbk_mask & (~stressed["related_transaction_id"].isin(all_ids))

    # Check 2: Are any agreed T+7 settlements flagged by the T05 settlement rule?
    ext_mask = stressed["transaction_id"].str.startswith("STRESS_EXT_", na=False)
    td_s = parse_mixed_date(stressed["trade_date"])
    sd_s = parse_mixed_date(stressed["settlement_date"])
    bday_lag = np.busday_count(
        td_s.fillna(pd.Timestamp("1970-01-01")).values.astype("datetime64[D]"),
        sd_s.fillna(pd.Timestamp("1970-01-01")).values.astype("datetime64[D]"),
    )
    has_note = stressed[note_col].notna() & (stressed[note_col].str.strip() != "")
    ext_cleared = td_s.notna() & sd_s.notna() & (bday_lag == 7) & has_note
    t05_flag = (
        td_s.notna()
        & sd_s.notna()
        & ((bday_lag < CONFIG["settle_min_days"]) | (bday_lag > CONFIG["settle_max_days"]))
        & (~ext_cleared)
    )
    ext_fp = ext_mask & t05_flag

    false_positives = int((rbk_dup_fp | rbk_link_fp | ext_fp).sum())

    return {
        "injected_defects": 210,
        "caught": caught,
        "missed": missed,
        "escape_rate_pct": round(100.0 * missed / 210.0, 1),
        "valid_lookalikes_tested": 120,
        "false_positives": false_positives,
    }

    # Evaluate detection on injected artifacts
    all_ids = set(stressed["transaction_id"])
    det_dups = stressed[stressed.duplicated("transaction_id", keep="first")]["transaction_id"].isin(
        set(dup_rows["transaction_id"])
    ).sum()
    det_orph = (
        (stressed["status"] == "CANCELLED")
        & (~stressed["related_transaction_id"].isin(all_ids))
        & stressed["transaction_id"].str.startswith("STRESS_ORPH_")
    ).sum()
    rev_grp = stressed[stressed["status"] == "REVERSED"].groupby("related_transaction_id").size()
    det_drev = sum(1 for oid in rev_rows_1["related_transaction_id"] if rev_grp.get(oid, 0) == 2)

    caught = int(det_dups + det_orph + det_drev)
    missed = 210 - caught
    return {
        "injected_defects": 210,
        "caught": caught,
        "missed": missed,
        "escape_rate_pct": round(100.0 * missed / 210.0, 1),
        "valid_lookalikes_tested": 120,
        "false_positives": 0,
    }


def verify_row_order_invariance(txn: pd.DataFrame) -> bool:
    """Confirms shuffling all 1.4M transaction rows produces identical aggregate counts."""
    shuffled = txn.sample(frac=1.0, random_state=99).reset_index(drop=True)
    orig_dups = int(txn.duplicated("transaction_id", keep="first").sum())
    shuf_dups = int(shuffled.duplicated("transaction_id", keep="first").sum())
    orig_nulls = int(txn["quantity"].isna().sum())
    shuf_nulls = int(shuffled["quantity"].isna().sum())
    assert orig_dups == shuf_dups and orig_nulls == shuf_nulls, "Row-order invariance violated!"
    return True


# =============================================================================
# 8. MAIN EXECUTION ENTRYPOINT
# =============================================================================
def main():
    t0 = time.perf_counter()
    CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

    tables = load_tables(CONFIG["data_dir"])
    m_res = check_masters(tables["accounts"], tables["securities"], tables["fx_rates"])
    t_res, txn_typed = check_transactions_and_pipeline(
        tables["transactions"], tables["pipeline_events"], tables["accounts"], tables["securities"]
    )
    p_res, snap_px = check_pricing_fx_ca(
        tables["prices"], tables["securities"], tables["fx_rates"], tables["corporate_actions"]
    )
    r_res = check_positions_ledger_exposure(
        txn_typed,
        tables["positions"],
        tables["comparison_positions"],
        tables["ledger_nav"],
        tables["securities"],
        tables["fx_rates"],
        snap_px,
    )

    domain_rates = {
        "accounts": m_res["acc_clean_rate"],
        "securities": m_res["sec_clean_rate"],
        "transactions": t_res["txn_clean_rate"],
        "pipeline": t_res["pip_clean_rate"],
        "positions": r_res["pos_clean_rate"],
        "pricing": p_res["prc_clean_rate"],
        "corporate_actions": p_res["ca_clean_rate"],
        "ledger": r_res["ldg_clean_rate"],
    }
    scores = compute_scores_and_sensitivity(domain_rates)
    stress = run_stress_test(tables["transactions"], tables["accounts"])
    invariance_ok = verify_row_order_invariance(tables["transactions"])

    summary = {
        "runtime_sec": round(time.perf_counter() - t0, 2),
        "row_order_invariant": invariance_ok,
        "masters": m_res,
        "transactions_and_pipeline": t_res,
        "pricing_fx_ca": p_res,
        "positions_and_ledger": r_res,
        "scores": scores,
        "stress_test": stress,
    }

    out_path = CONFIG["output_dir"] / "scorecard_and_results.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Completed full pipeline in {summary['runtime_sec']}s -> {out_path}")
    print(f"Platform Score: {scores['platform_score_pct']}% (Dirichlet 5th-95th: {scores['dirichlet_p05']}-{scores['dirichlet_p95']})")
    print(f"Stress Test: {stress['caught']}/{stress['injected_defects']} caught, {stress['false_positives']} false positives")


if __name__ == "__main__":
    main()

Completed full pipeline in 47.59s -> outputs/scorecard_and_results.json
Platform Score: 91.8% (Dirichlet 5th-95th: 84.7-96.8)
Stress Test: 180/210 caught, 0 false positives
